In [1]:
import json, polars as pl
from pathlib import Path
from IPython.display import Image, Markdown, display
ROOT = Path.cwd().parent
T = ROOT / "reports" / "tables"; F = ROOT / "reports" / "figures"
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_cols(20); pl.Config.set_tbl_width_chars(200)
def show(p): display(Image(filename=str(p)))
def tab(p, cols=None):
    d = pl.read_csv(p); return d.select(cols) if cols else d

# 04 · Feature ablation
A = price, B = +volume, C = +trade flow, D = +order-book bands, E = +funding/OI. Development walk-forward (7 folds) and the frozen holdout evaluation.

In [2]:
s = tab(T / 'model_summary.csv'); s.filter(pl.col('h') == 15).select('config','auc_mean','auc_sd','auc_min','folds_auc_gt_05','ll_improvement_mean','ic_mean')

config,auc_mean,auc_sd,auc_min,folds_auc_gt_05,ll_improvement_mean,ic_mean
str,f64,f64,f64,i64,f64,f64
"""lgbm_A_price_h15""",0.537509,0.008057,0.528073,7,0.001874,0.055893
"""logit_A_price_h15""",0.534362,0.007561,0.522695,7,0.001163,0.051448
"""rf_A_price_h15""",0.536892,0.007855,0.526184,7,0.001859,0.054873
"""lgbm_B_price_volume_h15""",0.537816,0.007742,0.529067,7,0.001882,0.056625
"""logit_B_price_volume_h15""",0.534593,0.00811,0.522026,7,0.001213,0.0519
"""rf_B_price_volume_h15""",0.537148,0.007904,0.52695,7,0.001917,0.055389
"""lgbm_C_plus_flow_h15""",0.537966,0.00711,0.529243,7,0.002023,0.056418
"""logit_C_plus_flow_h15""",0.534937,0.006901,0.523098,7,0.001299,0.052067
"""rf_C_plus_flow_h15""",0.53677,0.00795,0.526365,7,0.001901,0.055168


In [3]:
tab(T / 'dev_ablation_economics.csv')

config,tradable_auc,auc,sharpe_gross,total_return_gross,sharpe_base,total_return_base,breakeven_bps,n_trades,avg_trade_gross_bps
str,f64,f64,f64,f64,f64,f64,f64,i64,f64
"""lgbm_A_price_h15""",0.539591,0.537509,1.659726,1.158869,-13.675712,-0.99938,0.588459,212400,0.623008
"""lgbm_B_price_volume_h15""",0.539579,0.537816,2.057248,1.687023,-13.980427,-0.999505,0.700557,221529,0.747464
"""lgbm_C_plus_flow_h15""",0.539903,0.537966,2.244394,1.84506,-14.046295,-0.999383,0.747638,206458,0.836259
"""lgbm_D_plus_book_h15""",0.54044,0.539119,2.872199,2.987868,-14.682631,-0.999578,0.897847,239824,0.937035
"""lgbm_E_all_h15""",0.540296,0.538777,2.889033,3.32368,-14.093054,-0.999612,0.931522,248781,0.951546
"""logit_A_price_h15""",0.536723,0.534362,1.146717,0.646687,-8.253299,-0.990203,0.639439,127301,0.712449
"""logit_B_price_volume_h15""",0.537034,0.534593,1.541408,0.996733,-8.231357,-0.989239,0.830059,127551,0.933093
"""logit_C_plus_flow_h15""",0.537438,0.534937,1.660542,1.129859,-11.910787,-0.998463,0.650548,168514,0.765881
"""logit_D_plus_book_h15""",0.538599,0.536788,2.331264,2.026729,-13.699132,-0.999381,0.784986,213522,0.852705


## Holdout (frozen methodology)

In [4]:
p = T / 'holdout' / 'ablation.csv'
tab(p) if p.exists() else 'not yet evaluated'

model,fset,h,n,auc,log_loss,log_loss_baseline,ic_spearman,tradable_auc,sharpe_gross,total_return_gross,max_dd_gross,sharpe_base,total_return_base,max_dd_base,breakeven_bps
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""logit""","""A_price""",15,525540,0.534649,0.691675,0.693139,0.05023,0.538412,0.247668,0.035615,-0.270468,-7.060527,-0.864725,-0.868316,0.180246
"""lgbm""","""A_price""",15,525540,0.541159,0.69058,0.693139,0.058494,0.545334,1.860955,0.554877,-0.157782,-15.279454,-0.981562,-0.981723,0.591195
"""rf""","""A_price""",15,525540,0.542842,0.690364,0.693139,0.060641,0.5475,1.405247,0.441271,-0.166141,-19.769618,-0.996616,-0.996631,0.366899
"""logit""","""B_price_volume""",15,525540,0.533611,0.691825,0.693139,0.04853,0.537295,0.278435,0.044223,-0.278671,-7.125383,-0.867394,-0.870352,0.199284
"""lgbm""","""B_price_volume""",15,525540,0.540709,0.690649,0.693139,0.057379,0.54484,1.578163,0.44907,-0.158234,-15.707561,-0.983399,-0.983551,0.499475
"""rf""","""B_price_volume""",15,525540,0.542362,0.690428,0.693139,0.059775,0.547091,1.385615,0.403646,-0.174377,-20.575786,-0.99628,-0.996291,0.347692
"""logit""","""C_plus_flow""",15,525540,0.536273,0.691249,0.693139,0.050815,0.540531,0.893316,0.229091,-0.214449,-11.714188,-0.962268,-0.962481,0.377734
"""lgbm""","""C_plus_flow""",15,525540,0.540606,0.690645,0.693139,0.057332,0.544855,2.108098,0.629186,-0.14893,-16.561697,-0.984739,-0.984846,0.613904
"""rf""","""C_plus_flow""",15,525540,0.54266,0.690401,0.693139,0.060405,0.546967,1.407836,0.419621,-0.19277,-19.945212,-0.996213,-0.996231,0.35892
